In [9]:
from IPython.display import display, clear_output
import torch
import numpy as np
import time
import torch
from model import Model

model = Model()
model.load("./runs/latest/model.pth")

env = model.env
actor = model.actor

interactive = False
fps = 10
move_count = 0

actor_color = np.random.choice(["white", "black"])
print(f"Actor will play as {actor_color}")


observation shape: 218, action space: 29275
Model loaded from ./runs/latest/model.pth
Actor will play as black


In [11]:
while not env.board.is_game_over():
    clear_output(wait=True)
    display(env.board)
    
    if move_count == 0:
        obs = env.reset()
    
    # Check whose turn it is: chess.Board.turn is True for white, False for black
    if (env.board.turn and actor_color == "white") or (not env.board.turn and actor_color == "black"):
        # Actor's turn.
        obs = env.rollout(tensordict=obs, max_steps=1, policy=actor, auto_reset=False).get("next")[0]
    else:
        # Player's turn.
        legal_moves = env.get_legal_moves()
        if interactive:
            player_move = input("Enter your move: ")
        else:
            time.sleep(1 / fps)
            player_move = np.random.choice(legal_moves)
            
        if player_move == "exit":
            break
    
        action_index = legal_moves.index(player_move)
        action = torch.tensor(action_index)
        obs['action'] = action
        obs = env.rollout(tensordict=obs, max_steps=1, policy=None, auto_reset=False).get("next")[0]
    
    move_count += 1
print("Game over!")
print(f"Result: {env.board.result()}")

KeyboardInterrupt: 